# ABSA Training
Aspect-Based Sentiment Analysis on Arabic reviews using `aubmindlab/bert-base-arabertv2`

## 1. Imports

In [61]:
import re
import json
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

PyTorch version : 2.11.0+cpu
CUDA available  : False


## 2. Load Data

In [62]:
df = pd.read_excel("train_fixed.xlsx")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Shape: (1971, 9)
Columns: ['review_id', 'review_text', 'star_rating', 'date', 'business_name', 'business_category', 'platform', 'aspects', 'aspect_sentiments']


,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,2026-03-08 00:00:00,Noon,ecommerce,play_store,"[""app_experience"", ""delivery""]","{""app_experience"": ""negative"", ""delivery"": ""ne..."
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,قبل يومين (2),ممشي مصر Mawlana Cafe,كافيه,google_maps,"[""cleanliness"", ""ambiance"", ""service""]","{""cleanliness"": ""positive"", ""ambiance"": ""posit..."
2,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,1,قبل شهر,بيت لحم Beet Lahm,مطعم,google_maps,"[""service"", ""delivery"", ""food""]","{""service"": ""negative"", ""delivery"": ""negative""..."
3,3024,احلي مكان فزايد,5,قبل شهر,ذا بلكون كافيه الشيخ زايد,مطعم مأكولات ومشروبات,google_maps,"[""general""]","{""general"": ""positive""}"
4,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,4,قبل سنة,The Best Restaurant,مطعم,google_maps,"[""food"", ""price""]","{""food"": ""positive"", ""price"": ""positive""}"


## 3. Parse JSON Columns

In [63]:
df["aspects"] = df["aspects"].apply(json.loads)
df["aspect_sentiments"] = df["aspect_sentiments"].apply(json.loads)
print("JSON columns parsed successfully.")

JSON columns parsed successfully.


## 4. Explode to One Row per Aspect

In [64]:
rows = []
for _, row in df.iterrows():
    review = row["review_text"]
    for aspect, sentiment in row["aspect_sentiments"].items():
        rows.append({
            "text": review,
            "aspect": aspect,
            "label": sentiment
        })

train_df = pd.DataFrame(rows)
print(f"Total rows after explosion: {len(train_df)}")
train_df.head()

Total rows after explosion: 3333


,text,aspect,label
0,لا يوجد الدفع بالبطاقه عند الاستلام,app_experience,negative
1,لا يوجد الدفع بالبطاقه عند الاستلام,delivery,negative
2,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,cleanliness,positive
3,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,ambiance,positive
4,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,service,positive


## 5. Label Encoding

In [65]:
LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}

train_df["label_id"] = train_df["label"].map(LABEL_MAP)

# Sanity check — catch any unmapped labels
assert train_df["label_id"].notna().all(), f"Unmapped labels found: {train_df[train_df['label_id'].isna()]['label'].unique()}"

print(train_df[["label", "label_id"]].value_counts().sort_index())

label     label_id
negative  0           1538
neutral   1            149
positive  2           1646
Name: count, dtype: int64


## 6. Text Cleaning

In [66]:
def clean_text(text: str) -> str:
    text = str(text)

    # Remove newlines
    text = re.sub(r"\n+", " ", text)

    # Remove dates
    text = re.sub(r"\d{4}-\d{2}-\d{2}", "", text)
    text = re.sub(r"\d{2}/\d{2}/\d{4}", "", text)

    # Underscore → space
    text = text.replace("_", " ")

    # Remove special symbols
    text = re.sub(r"[@#$%^&*+!~|?><\-]", "", text)

    # Remove repeated dots
    text = re.sub(r"\.{2,}", "", text)

    # Remove non-Arabic, non-word characters
    text = re.sub(r"[^\w\s\u0600-\u06FF]", "", text)

    # Normalize Arabic letters
    text = re.sub(r"[أإآ]", "ا", text)
    text = re.sub(r"ة", "ه", text)

    # Collapse repeated characters (max 2)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    text = text.replace("٠","0").replace("١","1").replace("٢","2")\
               .replace("٣","3").replace("٤","4").replace("٥","5")\
               .replace("٦","6").replace("٧","7").replace("٨","8").replace("٩","9")

    # Clean whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply cleaning
train_df["text_clean"] = train_df["text"].apply(clean_text)

# Preview before vs after
print("Before:", train_df["text"].iloc[0][:100])
print("After :", train_df["text_clean"].iloc[0][:100])

Before: لا يوجد الدفع بالبطاقه عند الاستلام
After : لا يوجد الدفع بالبطاقه عند الاستلام


In [67]:
print(train_df[[ "label_id"]].value_counts())

label_id
2           1646
0           1538
1            149
Name: count, dtype: int64


In [68]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. جهزي input
train_df["input"] = train_df["text"] + " " + train_df["aspect"]

# 2. labels (حوّليها أرقام)
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}
train_df["label_id"] = train_df["label"].map(label_map)

# 3. split
X_train, X_val, y_train, y_val = train_test_split(
    train_df["input"],
    train_df["label_id"],
    test_size=0.2,
    random_state=42
)

# 4. TF-IDF (تحويل الكلام لأرقام)
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

# 5. Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# 6. prediction
y_pred = model.predict(X_val_vec)

# 7. evaluation
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.93      0.92       320
           1       1.00      0.29      0.44        28
           2       0.90      0.93      0.92       319

    accuracy                           0.91       667
   macro avg       0.94      0.72      0.76       667
weighted avg       0.91      0.91      0.90       667



## 7. Build Model Input  
Format: `<review> [SEP] <aspect>` — the `[SEP]` token gives the model a clear boundary between review and aspect.

In [69]:
train_df["input"] = train_df["text_clean"] + " [SEP] " + train_df["aspect"]

print(f"Sample input:\n{train_df['input'].iloc[0]}")

Sample input:
لا يوجد الدفع بالبطاقه عند الاستلام [SEP] app_experience


## 8. Train / Validation Split

In [70]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["label_id"]   # keep class balance in both splits
)

train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)

print(f"Train : {len(train_data)}")
print(f"Val   : {len(val_data)}")
print("\nTrain label distribution:")
print(train_data["label"].value_counts())

Train : 2666
Val   : 667

Train label distribution:
label
positive    1317
negative    1230
neutral      119
Name: count, dtype: int64


## 9. Load Tokenizer & Model

In [71]:
MODEL_NAME = "aubmindlab/bert-base-arabertv2"
NUM_LABELS = 3
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True
)

print("Model loaded successfully ✅")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully ✅
Parameters: 135,195,651


## 10. Dataset Class

In [72]:
class ABSADataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int = MAX_LEN):
        self.texts     = df["input"].tolist()
        self.labels    = df["label_id"].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids"      : encoding["input_ids"].squeeze(),
            "attention_mask" : encoding["attention_mask"].squeeze(),
            "labels"         : torch.tensor(self.labels[idx], dtype=torch.long)
        }


train_dataset = ABSADataset(train_data, tokenizer)
val_dataset   = ABSADataset(val_data,   tokenizer)

# Quick sanity check on a single sample
sample = train_dataset[0]
print(f"input_ids shape     : {sample['input_ids'].shape}")
print(f"attention_mask shape: {sample['attention_mask'].shape}")
print(f"label               : {sample['labels'].item()}")

input_ids shape     : torch.Size([128])
attention_mask shape: torch.Size([128])
label               : 0


In [73]:
!pip install langdetect

In [74]:
from langdetect import detect
from collections import Counter

langs = []

for text in train_df["text"]:
    try:
        langs.append(detect(str(text)))
    except:
        langs.append("unknown")

print(Counter(langs))

Counter({'ar': 3180, 'fa': 53, 'en': 38, 'ur': 38, 'so': 11, 'ro': 4, 'nl': 3, 'pl': 3, 'unknown': 2, 'de': 1})


In [75]:
!pip install "accelerate>=1.1.0" --quiet

In [76]:
pip uninstall transformers accelerate -y

Found existing installation: transformers 5.6.2
Uninstalling transformers-5.6.2:
  Successfully uninstalled transformers-5.6.2
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Note: you may need to restart the kernel to use updated packages.


In [77]:
pip install -U transformers accelerate datasets torch

  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.6.2-py3-none-any.whl (10.4 MB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)

   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers

In [78]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./arabert-absa",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
c:\Users\dalia\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.400185,0.338748,0.907046,0.694814
2,0.308122,0.311659,0.910045,0.793433
3,0.252128,0.268821,0.913043,0.795463
4,0.206737,0.301743,0.902549,0.777490
5,0.158271,0.323718,0.899550,0.772187


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\dalia\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

c:\Users\dalia\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\dalia\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=835, training_loss=0.2744994186355682, metrics={'train_runtime': 6489.1531, 'train_samples_per_second': 2.054, 'train_steps_per_second': 0.129, 'total_flos': 876825464578560.0, 'train_loss': 0.2744994186355682, 'epoch': 5.0})

In [79]:
results = trainer.evaluate(eval_dataset=val_dataset)

print("===== VALIDATION RESULTS =====")
print(f"Accuracy  : {results['eval_accuracy']:.4f}")
print(f"F1 Macro  : {results['eval_f1_macro']:.4f}")
print(f"Loss      : {results['eval_loss']:.4f}")

c:\Users\dalia\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.158271,0.268821,5,0.913043,0.795463


===== VALIDATION RESULTS =====
Accuracy  : 0.9130
F1 Macro  : 0.7955
Loss      : 0.2688


In [81]:
import pandas as pd
import json

# 1. قراءة الملف
validation_df = pd.read_excel("validation_fixed.xlsx")

# 2. تحويل JSON strings لو موجودة
validation_df["aspect_sentiments"] = validation_df["aspect_sentiments"].apply(json.loads)

# 3. تفكيك نفس طريقة التدريب (one row per aspect)
rows = []

for _, row in validation_df.iterrows():
    review = row["review_text"]
    sentiments = row["aspect_sentiments"]

    for aspect, sentiment in sentiments.items():
        rows.append({
            "text": review,
            "aspect": aspect,
            "label": sentiment
        })

validation_df = pd.DataFrame(rows)

# 4. label mapping (نفس التدريب)
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

validation_df["label_id"] = validation_df["label"].map(label_map)

# 5. build input (نفس التدريب)
validation_df["input"] = validation_df["text"] + " [SEP] " + validation_df["aspect"]

# 6. Dataset
validation_dataset = ABSADataset(validation_df, tokenizer)

# 7. Evaluate
results = trainer.evaluate(eval_dataset=validation_dataset)

print("=====  Official VALIDATION RESULTS =====")
print(results)



c:\Users\dalia\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.158271,0.285889,5,0.910714,0.769814


=====  Official VALIDATION RESULTS =====
{'eval_loss': 0.28588926792144775, 'eval_accuracy': 0.9107142857142857, 'eval_f1_macro': 0.7698135732660875}


In [83]:

import pandas as pd
import torch
import numpy as np

df = pd.read_excel("unlabeled_fixed.xlsx")

# aspects لازم تكون ثابتة
aspects = ["food", "service", "price", "cleanliness", "delivery"]

rows = []

for _, row in df.iterrows():
    review_id = row["review_id"]
    review_text = str(row["review_text"])

    for aspect in aspects:
        rows.append({
            "review_id": review_id,
            "review_text": review_text,
            "aspect": aspect,
            "input": review_text + " [SEP] " + aspect
        })

pred_df = pd.DataFrame(rows)

In [114]:
import pandas as pd
import torch
import numpy as np

# 1. Load unlabeled file
unlabeled_df = pd.read_excel("DeepX_hidden_test.xlsx")

# 2. Explode per aspect
aspects = ["food", "service", "price", "cleanliness", "delivery"]

rows = []
for _, row in unlabeled_df.iterrows():
    review_id = row["review_id"]
    review_text = str(row["review_text"])
    for aspect in aspects:
        rows.append({
            "review_id": review_id,
            "review_text": review_text,
            "aspect": aspect,
            "input": review_text + " [SEP] " + aspect
        })

pred_df = pd.DataFrame(rows)

# 3. Predict in batches
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BATCH_SIZE = 16  # لو لسه بيواجه مشكلة، خليها 8
all_preds = []

inputs_list = list(pred_df["input"])

for i in range(0, len(inputs_list), BATCH_SIZE):
    batch_texts = inputs_list[i : i + BATCH_SIZE]
    
    encodings = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model(
            input_ids=encodings["input_ids"],
            attention_mask=encodings["attention_mask"]
        )
    
    preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
    all_preds.extend(preds)
    
    # عشان تشوف البرogress
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"Processed {i + len(batch_texts)} / {len(inputs_list)}")

# 4. Map labels back
label_map_inv = {0: "negative", 1: "neutral", 2: "positive"}
pred_df["prediction"] = [label_map_inv[p] for p in all_preds]

# 5. Save
pred_df.to_excel("predictions.xlsx", index=False)
print("✅ Predictions saved to predictions.xlsx")
print(pred_df[["review_id", "review_text", "aspect", "prediction"]].head(10))


Processed 16 / 2500
Processed 176 / 2500
Processed 336 / 2500
Processed 496 / 2500
Processed 656 / 2500
Processed 816 / 2500
Processed 976 / 2500
Processed 1136 / 2500
Processed 1296 / 2500
Processed 1456 / 2500
Processed 1616 / 2500
Processed 1776 / 2500
Processed 1936 / 2500
Processed 2096 / 2500
Processed 2256 / 2500
Processed 2416 / 2500
✅ Predictions saved to predictions.xlsx
   review_id                                        review_text       aspect  \
0         23  جئت خصيصا لتجربه طبق طاجن الملوخيه باللحم مع ص...         food   
1         23  جئت خصيصا لتجربه طبق طاجن الملوخيه باللحم مع ص...      service   
2         23  جئت خصيصا لتجربه طبق طاجن الملوخيه باللحم مع ص...        price   
3         23  جئت خصيصا لتجربه طبق طاجن الملوخيه باللحم مع ص...  cleanliness   
4         23  جئت خصيصا لتجربه طبق طاجن الملوخيه باللحم مع ص...     delivery   
5         25  أسوأ مطعم\nطلبت منه اوردر من الساعة 2 المفروض ...         food   
6         25  أسوأ مطعم\nطلبت منه اوردر من الساعة 2 المف

In [115]:
print(model.config._name_or_path)

aubmindlab/bert-base-arabertv2


In [116]:
trainer.save_model("./arabert-absa")
tokenizer.save_pretrained("./arabert-absa")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./arabert-absa\\tokenizer_config.json', './arabert-absa\\tokenizer.json')

In [127]:
import json

submission = []
for review_id, group in pred_df.groupby("review_id", sort=False):
    submission.append({
        "review_id": int(review_id),
        "aspects": group["aspect"].tolist(),
        "aspect_sentiments": dict(zip(group["aspect"], group["prediction"]))
    })

with open("submission.json", "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

print(f"Saved {len(submission)} reviews to submission.json")
print(json.dumps(submission[:10], ensure_ascii=False, indent=2))

Saved 500 reviews to submission.json
[
  {
    "review_id": 23,
    "aspects": [
      "food",
      "service",
      "price",
      "cleanliness",
      "delivery"
    ],
    "aspect_sentiments": {
      "food": "positive",
      "service": "negative",
      "price": "negative",
      "cleanliness": "negative",
      "delivery": "negative"
    }
  },
  {
    "review_id": 25,
    "aspects": [
      "food",
      "service",
      "price",
      "cleanliness",
      "delivery"
    ],
    "aspect_sentiments": {
      "food": "negative",
      "service": "negative",
      "price": "negative",
      "cleanliness": "negative",
      "delivery": "negative"
    }
  },
  {
    "review_id": 65,
    "aspects": [
      "food",
      "service",
      "price",
      "cleanliness",
      "delivery"
    ],
    "aspect_sentiments": {
      "food": "positive",
      "service": "positive",
      "price": "positive",
      "cleanliness": "positive",
      "delivery": "positive"
    }
  },
  {
    "review_